# 동일 조건 반복성 보고서

같은 조건에서 생성한 여러 answers의 **평가 분포**를 비교합니다. 전체 문항과 개별 문항의 등급 건수·비율을 선택해 볼 수 있으며, 회차별 개별 판정은 원본 agreement에 보존합니다.


In [ ]:
import os

import mlflow
import pandas as pd
import panel as pn
from IPython.display import display

from elice_reporting import export_html, find_root, from_files, from_mlflow, report_purpose_callout
from elice_reporting.efficiency_sections import efficiency_view
from elice_reporting.repeatability_sections import (
    repeatability_conditions,
    repeatability_metric_guide,
)
from elice_reporting.repeatability_view import change_heatmap, quality_heatmaps

pn.extension()
ROOT = find_root()

In [ ]:
# 보고서마다 이 셀만 수정합니다. 실제 주소와 인증정보는 환경변수에 둡니다.
SOURCE_MODE = "mlflow"  # mlflow | local
RUN_ID = "a3a3123c03fe4905928f56130b68488b"
REPORT_TITLE = "Terra · Serverless 임베딩 — 반복성"
OUTPUT_HTML = ROOT / "artifacts/reports/terra-serverless-a1-a2-repeatability.html"
LOCAL_REPORT = ROOT / "artifacts/evaluation/repeatability-result.json"
LOCAL_INPUTS = ROOT / "artifacts/evaluation/repeatability-inputs.json"

In [ ]:
if SOURCE_MODE == "mlflow":
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    client = mlflow.MlflowClient()
    if not RUN_ID:
        experiment = client.get_experiment_by_name(
            os.environ.get("QA_MLFLOW_EXPERIMENT_NAME", "elice-techblog-rag-evaluation")
        )
        if experiment is None:
            raise ValueError("실험을 찾을 수 없습니다. 환경 설정을 확인하세요.")
        condition = "tags.`evaluation.kind` = 'repeatability' AND attributes.status = 'FINISHED'"
        runs = client.search_runs(
            [experiment.experiment_id],
            filter_string=condition,
            max_results=20,
        )
        display(
            pd.DataFrame(
                [{"run_id": r.info.run_id, "name": r.data.tags.get("mlflow.runName")} for r in runs]
            )
        )
        raise ValueError("위 목록에서 RUN_ID를 설정하고 다시 실행하세요.")
    data = from_mlflow("repeatability", RUN_ID, ROOT / "artifacts/reports/cache")
elif SOURCE_MODE == "local":
    data = from_files("repeatability", LOCAL_REPORT, LOCAL_INPUTS)
else:
    raise ValueError("SOURCE_MODE는 mlflow 또는 local이어야 합니다.")

## 1. 평가 조건

Agreement 보고서와 같은 범주별 설정을 표시합니다. Dataset·답변/검색 설정·Corpus/청킹·임베딩/인덱스·Agent·생성 환경과 원본 Run 링크를 확인할 수 있습니다.


In [ ]:
conditions_view = repeatability_conditions(data)

## 2. 지표 정의와 판정 기준

원본 agreement와 같은 rubric 등급 기준을 보여주고, 경계 충족률·시행 간 평균 변화폭의 계산 및 제외 규칙을 설명합니다.


## 3. 시행 간 평균 변화폭

질문을 대응시켜 계산한 두 경계의 평균 변화폭을 히트맵으로 보여줍니다. 평가 주체 드롭다운에서 전체 모델 평균 또는 개별 주체를 선택하고, 셀 hover에서 비교 가능 건수를 확인할 수 있습니다. 세로 점선 오른쪽은 전체 항목 평균입니다.


## 4. 문항별 품질 수준

질문 선택에서 전체 문항 또는 개별 질문을 고릅니다. 전체 문항에는 유효 문항만 같은 비중으로 평균한 행을, 오른쪽에는 유효 평가 항목을 같은 비중으로 평균한 열을 추가합니다. 세로 점선 오른쪽이 항목 평균이고 가로 점선 아래가 문항 평균입니다. Gold의 두 rubric 경계를 나란한 히트맵에 표시합니다. 기본 `전체 모델 평균`은 문항·평가 항목별 모델의 시행 평균 충족률을 같은 비중으로 평균하고, 평가 주체를 고르면 그 주체의 값만 표시합니다. 사람 판정은 모델 평균에 포함하지 않습니다. 개별 시행의 등급과 사유는 평가 조건의 원본 agreement 링크에서 확인할 수 있습니다.


In [ ]:
definitions_view = repeatability_metric_guide(data)
change_view = change_heatmap(data)
quality = quality_heatmaps(data)

In [ ]:
report_view = pn.Column(
    pn.pane.Markdown("# " + REPORT_TITLE),
    report_purpose_callout("repeatability"),
    pn.pane.Markdown("## 1. 평가 조건"),
    conditions_view,
    pn.pane.Markdown("## 2. 지표 정의와 판정 기준"),
    definitions_view,
    pn.pane.Markdown("## 3. 시행 간 평균 변화폭"),
    change_view,
    pn.pane.Markdown("## 4. 문항별 품질 수준"),
    quality,
    pn.pane.Markdown("## 5. 효율성"),
    efficiency_view(data),
    sizing_mode="stretch_width",
    max_width=1180,
    styles={"margin": "0 auto"},
)
report_view

## HTML export

저장된 결과만 로컬 HTML로 내보냅니다.


In [ ]:
html_path = export_html(report_view, OUTPUT_HTML, title=REPORT_TITLE, overwrite=True)
html_path